<a href="https://colab.research.google.com/github/Sangam1293/Sangam216/blob/main/Cradit_Card_Fraud_Deaction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

In [16]:
# Load dataset
df = pd.read_csv("creditcard.csv")
print(df.head)

<bound method NDFrame.head of             Time        V1        V2        V3        V4        V5        V6  \
0            0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388   
1            0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361   
2            1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499   
3            1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203   
4            2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921   
...          ...       ...       ...       ...       ...       ...       ...   
233738  147684.0  1.938933 -0.726856 -0.197019  0.670379 -1.043038 -0.473694   
233739  147684.0 -0.458365 -0.734428 -1.299382 -1.498641 -1.073566  0.340757   
233740  147685.0  2.072601 -0.169893 -1.165646  0.150432  0.120419 -0.468132   
233741  147685.0  2.302987 -1.402751 -0.661621 -1.441633 -1.615912 -1.008410   
233742  147686.0  1.406398 -1.009139 -3.381354  0.247401  1.245001 -0.374091   

         

In [17]:
print("Shape of dataset:")
print(df.shape)

Shape of dataset:
(233743, 31)


In [18]:
# Check null values
print("Null values before cleaning:")
print(df.isnull().sum())

# Drop null values
df = df.dropna()

Null values before cleaning:
Time      0
V1        0
V2        0
V3        0
V4        0
V5        0
V6        0
V7        0
V8        0
V9        0
V10       0
V11       0
V12       0
V13       0
V14       1
V15       1
V16       1
V17       1
V18       1
V19       1
V20       1
V21       1
V22       1
V23       1
V24       1
V25       1
V26       1
V27       1
V28       1
Amount    1
Class     1
dtype: int64


In [19]:
# Check again
print("\nNull values after cleaning:")
print(df.isnull().sum())


Null values after cleaning:
Time      0
V1        0
V2        0
V3        0
V4        0
V5        0
V6        0
V7        0
V8        0
V9        0
V10       0
V11       0
V12       0
V13       0
V14       0
V15       0
V16       0
V17       0
V18       0
V19       0
V20       0
V21       0
V22       0
V23       0
V24       0
V25       0
V26       0
V27       0
V28       0
Amount    0
Class     0
dtype: int64


In [20]:
print("\nClass Distribution:")
print(df["Class"].value_counts())

print("\nClass Distribution (%):")
print(df["Class"].value_counts(normalize=True) * 100)


Class Distribution:
Class
0.0    233319
1.0       423
Name: count, dtype: int64

Class Distribution (%):
Class
0.0    99.819031
1.0     0.180969
Name: proportion, dtype: float64


In [21]:
# Separate features and target
X = df.drop("Class", axis=1)
y = df["Class"]
print("\nFeature Shape:")
print(X.shape)

print("\nTarget Shape:")
print(y.shape)


Feature Shape:
(233742, 30)

Target Shape:
(233742,)


In [22]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [23]:
# Scale data for SVM
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [24]:
# Apply SMOTE only to training data
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_scaled,
    y_train
)

print("Before SMOTE:")
print(y_train.value_counts())

print("\nAfter SMOTE:")
print(y_train_smote.value_counts())

Before SMOTE:
Class
0.0    186655
1.0       338
Name: count, dtype: int64

After SMOTE:
Class
0.0    186655
1.0    186655
Name: count, dtype: int64


In [25]:
model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss"
)

model.fit(X_train_smote, y_train_smote)

# Fraud probability
y_prob = model.predict_proba(X_test_scaled)[:, 1]
print("\nFirst 10 probabilities:")
print(y_prob[:10])

print("\nProbability Information:")

print(
    "Minimum probability:",
    y_prob.min()
)

print(
    "Maximum probability:",
    y_prob.max()
)

print(
    "Mean probability:",
    y_prob.mean()
)


# Threshold tuning
threshold = 0.30

y_pred = (y_prob >= threshold).astype(int)

print("\nXGBoost Results")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("ROC-AUC:", roc_auc_score(y_test, y_prob))


First 10 probabilities:
[0.00734102 0.09905253 0.00121156 0.00339847 0.00142027 0.02982935
 0.01038699 0.00101291 0.00489487 0.02193717]

Probability Information:
Minimum probability: 6.3734806e-05
Maximum probability: 0.99960846
Mean probability: 0.036361147

XGBoost Results
              precision    recall  f1-score   support

         0.0       1.00      0.98      0.99     46664
         1.0       0.08      0.95      0.15        85

    accuracy                           0.98     46749
   macro avg       0.54      0.97      0.57     46749
weighted avg       1.00      0.98      0.99     46749

Confusion Matrix:
[[45720   944]
 [    4    81]]
ROC-AUC: 0.9813306138502033


In [26]:
for threshold in [0.1, 0.2, 0.3, 0.4, 0.5]:

    pred = (y_prob >= threshold).astype(int)

    print(
        threshold,
        "Precision:", precision_score(y_test, pred),
        "Recall:", recall_score(y_test, pred),
        "F1:", f1_score(y_test, pred)
    )

0.1 Precision: 0.02165302350145234 Recall: 0.9647058823529412 F1: 0.042355371900826444
0.2 Precision: 0.04723032069970846 Recall: 0.9529411764705882 F1: 0.09
0.3 Precision: 0.07902439024390244 Recall: 0.9529411764705882 F1: 0.14594594594594595
0.4 Precision: 0.1187683284457478 Recall: 0.9529411764705882 F1: 0.21121251629726207
0.5 Precision: 0.17270788912579957 Recall: 0.9529411764705882 F1: 0.2924187725631769


In [ ]:
svm = SVC(
    kernel="rbf",
    probability=True,
    random_state=42
)

svm.fit(X_train_smote, y_train_smote)

svm_prob = svm.predict_proba(X_test_scaled)[:, 1]

svm_pred = (svm_prob >= 0.5).astype(int)

print("\nSVM Results")
print(classification_report(y_test, svm_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, svm_pred))

print("ROC-AUC:", roc_auc_score(y_test, svm_prob))

In [ ]:
importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("\nTop Features:")
print(importance.head(10))

importance.head(10).plot(kind="bar")

plt.title("Top 10 Feature Importance - XGBoost")
plt.xlabel("Features")
plt.ylabel("Importance")
plt.show()